# 06 permissions

Question: does `golem-permissions` read its configuration as strictly as
`okf/plugins/permissions.md` says, decide every check by the rules of that page in their order,
and does the seed `config/permissions.toml` grant a player of the group default what the group
default of LuckPerms on the live server grants, and nothing of the admin commands.

`golem-devtool permissions` prints the decision of a configuration for a player and nodes, with
the reason that names the rule, and `golem-devtool permissions-groups` prints the groups of a
player in the order a check searches them. The notebook writes small configurations into a
temporary directory and holds every decision, every reason and every refusal to the expectation
written in its cell. The expectations for the seed come from the wiki of the server,
`okf/modpack/plugins.md` of `skytekx3-1201-server`, section Permissions, which lists the 39 nodes
of the group default. The last section times a check.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The seed of the permissions, repository relative.
seed = "config/permissions.toml"

# How often the cost section runs every check, and the bound of one check in nanoseconds. The
# devtool is a debug build, in which a check takes a few microseconds, so the bound only catches
# a check that searches the configuration instead of its tables.
repeat = 20000
max_nanos_per_check = 50000

# The size of the large configuration of the cost section, and how much slower a check in it
# may be than the same check in a configuration with one player.
large_groups = 200
large_players = 5000
max_growth = 10

# The length of a chain of inheritance that loading must take without overflowing its stack.
long_chain = 100000

## Setup

The cell finds the repository, builds `golem-devtool` and prints the versions of Python, cargo
and rustc. The helpers write a configuration into a temporary directory, run one subcommand on
it and compare a list of cases with what it printed. The players are made up UUIDs.

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-06-"))
SEED = P.root / seed

run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
assert DEVTOOL.is_file(), "cargo built no golem-devtool"

ALEX = "a1e5a1e5-0000-4000-8000-000000000001"
SAM = "5a305a30-0000-4000-8000-000000000002"
NOBODY = "00000000-0000-4000-8000-0000000000ff"
NAMES = {ALEX: "Alex", SAM: "Sam", NOBODY: "nobody"}
COUNTS = {"decisions": 0, "refusals": 0, "group lists": 0}
_written = [0]


def write(text):
    _written[0] += 1
    path = SCRATCH / ("config-%03d.toml" % _written[0])
    path.write_text(text, encoding="utf-8")
    return path


def devtool(*arguments):
    return run([DEVTOOL, *map(str, arguments)], cwd=P.root, timeout=600)


def decide(config, player, nodes, extra=()):
    path = config if isinstance(config, Path) else write(config)
    arguments = ["permissions", "--config", path, "--player", player]
    for node in nodes:
        arguments += ["--node", node]
    result = devtool(*arguments, *extra)
    assert result.returncode == 0, "permissions exited %d\n%s" % (result.returncode, result.stderr)
    return json.loads(result.stdout)


def judge(config, cases):
    # Each case is (player, node, value, reason). The cell prints one row per case and asserts
    # that every decision matches.
    path = config if isinstance(config, Path) else write(config)
    got = {}
    for player in dict.fromkeys(case[0] for case in cases):
        nodes = [case[1] for case in cases if case[0] == player]
        decisions = decide(path, player, nodes)["decisions"]
        assert [d["node"] for d in decisions] == nodes, decisions
        for decision in decisions:
            got[(player, decision["node"])] = decision
    wrong = []
    for player, node, value, reason in cases:
        decision = got[(player, node)]
        ok = decision["value"] is value and decision["reason"] == reason
        COUNTS["decisions"] += 1
        print("%-3s %-7s %-28s %-5s %s" % ("ok" if ok else "BAD", NAMES.get(player, player)[:7], node, decision["value"], decision["reason"]))
        if not ok:
            wrong.append("%s %s: expected %s %r, got %s %r" % (player, node, value, reason, decision["value"], decision["reason"]))
    assert not wrong, "\n".join(wrong)


def groups_of(config, player):
    path = config if isinstance(config, Path) else write(config)
    result = devtool("permissions-groups", "--config", path, "--player", player)
    assert result.returncode == 0, "permissions-groups exited %d\n%s" % (result.returncode, result.stderr)
    COUNTS["group lists"] += 1
    return [(g["group"], g["weight"], g["depth"], g["rank"]) for g in json.loads(result.stdout)["groups"]]


def refusal(text):
    path = write(text)
    result = devtool("permissions", "--config", path, "--player", NOBODY, "--node", "essentials.home")
    prefix = "golem-devtool: permissions: %s: " % path
    assert result.returncode == 1, "the configuration was taken:\n%s\n%s" % (text, result.stdout)
    assert result.stdout == "", "a refused configuration printed on standard output: %r" % result.stdout
    assert result.stderr.startswith(prefix) and result.stderr.endswith("\n"), result.stderr
    return result.stderr[len(prefix):-1]


print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("rustc:", run(["rustc", "--version"], cwd=P.root, check=True).stdout.strip())
print("seed:", P.rel(SEED))

## Exact nodes and wildcards

A node names itself, `prefix.*` names every node below the prefix at any depth but not the prefix
itself, as in LuckPerms, and `*` names every node. An exact deny takes one node out of a wildcard
grant, and the nodes below that node still fall under the wildcard.

In [ ]:
WILDCARDS = '''
[groups.default]
nodes = [
    "essentials.home",
    "essentials.warp.*",
    "-essentials.warp.secret",
    "essentials.signs.use.*",
]
'''
judge(WILDCARDS, [
    (NOBODY, "essentials.home", True, "Group default grants essentials.home"),
    (NOBODY, "essentials.warp.list", True, "Group default grants essentials.warp.*"),
    (NOBODY, "essentials.warp.list.all", True, "Group default grants essentials.warp.*"),
    (NOBODY, "essentials.warp", None, "No rule names essentials.warp"),
    (NOBODY, "essentials.warp.secret", False, "Group default denies essentials.warp.secret"),
    (NOBODY, "essentials.warp.secret.room", True, "Group default grants essentials.warp.*"),
    (NOBODY, "essentials.signs.use.buy", True, "Group default grants essentials.signs.use.*"),
    (NOBODY, "essentials.signs.use", None, "No rule names essentials.signs.use"),
    (NOBODY, "essentials.fly", None, "No rule names essentials.fly"),
])

## The longest wildcard wins

Within one source the exact node decides first, then the wildcard with the longest prefix, then
`*`. The group below grants everything, denies Essentials, grants the homes again and denies the
secret homes, and every node lands on the most specific of them.

In [ ]:
LONGEST = '''
[groups.default]
nodes = ["*", "-essentials.*", "essentials.home.*", "-essentials.home.secret.*"]
'''
judge(LONGEST, [
    (NOBODY, "minecraft.command.give", True, "Group default grants *"),
    (NOBODY, "fly", True, "Group default grants *"),
    (NOBODY, "essentials.fly", False, "Group default denies essentials.*"),
    (NOBODY, "essentials.home", False, "Group default denies essentials.*"),
    (NOBODY, "essentials.home.bed", True, "Group default grants essentials.home.*"),
    (NOBODY, "essentials.home.secret", True, "Group default grants essentials.home.*"),
    (NOBODY, "essentials.home.secret.cave", False, "Group default denies essentials.home.secret.*"),
])

## Specificity over the order of the file

The order in which a source lists its nodes plays no part. Two groups with the same nodes in
opposite orders decide every node the same way.

In [ ]:
FORWARD = '''
[groups.default]
nodes = ["-essentials.*", "essentials.home"]
'''
BACKWARD = '''
[groups.default]
nodes = ["essentials.home", "-essentials.*"]
'''
for config in (FORWARD, BACKWARD):
    judge(config, [
        (NOBODY, "essentials.home", True, "Group default grants essentials.home"),
        (NOBODY, "essentials.fly", False, "Group default denies essentials.*"),
    ])

## A deny wins over a grant at an equal specificity

When one source grants and denies the same node, which is an equal specificity, the deny wins, in
a group as in the nodes of a player, for a node, a wildcard and `*`.

In [ ]:
EQUAL = '''
[groups.default]
nodes = ["essentials.home", "-essentials.home", "essentials.warp.*", "-essentials.warp.*", "*", "-*"]

[players.%s]
nodes = ["essentials.fly", "-essentials.fly"]
''' % ALEX
judge(EQUAL, [
    (NOBODY, "essentials.home", False, "Group default grants and denies essentials.home, and the deny wins"),
    (NOBODY, "essentials.warp.list", False, "Group default grants and denies essentials.warp.*, and the deny wins"),
    (NOBODY, "minecraft.command.give", False, "Group default grants and denies *, and the deny wins"),
    (ALEX, "essentials.fly", False, "The player grants and denies essentials.fly, and the deny wins"),
])

## The player before the groups

The nodes of the player decide first, whatever the groups say. A wildcard deny of the player takes
away a node that a group grants by name, which is where Golem differs from LuckPerms on purpose.
A node the player does not name falls through to the groups.

In [ ]:
PLAYER = '''
[groups.default]
nodes = ["essentials.home", "essentials.suicide", "-essentials.fly"]

[players.%s]
nodes = ["-essentials.suicide", "essentials.fly"]

[players.%s]
nodes = ["-essentials.*"]
''' % (ALEX, SAM)
judge(PLAYER, [
    (ALEX, "essentials.suicide", False, "The player denies essentials.suicide"),
    (ALEX, "essentials.fly", True, "The player grants essentials.fly"),
    (ALEX, "essentials.home", True, "Group default grants essentials.home"),
    (SAM, "essentials.home", False, "The player denies essentials.*"),
    (SAM, "minecraft.command.give", None, "No rule names minecraft.command.give"),
    (NOBODY, "essentials.suicide", True, "Group default grants essentials.suicide"),
    (NOBODY, "essentials.fly", False, "Group default denies essentials.fly"),
])

## The order of weight

The groups are searched in ranks, a higher weight first, and the first rank in which any group
matches decides, even when a group of a lower rank names the node more specifically. Weight comes
before depth, so a group inherited at depth 2 with a higher weight comes before the group that
inherits it.

In [ ]:
WEIGHT = '''
[groups.default]
nodes = ["essentials.home", "-essentials.fly"]

[groups.vip]
weight = 10
nodes = ["essentials.fly", "-essentials.*"]

[groups.staff]
weight = 20
nodes = ["essentials.home"]

[players.%s]
groups = ["vip"]

[players.%s]
groups = ["vip", "staff"]
''' % (ALEX, SAM)
judge(WEIGHT, [
    (ALEX, "essentials.fly", True, "Group vip grants essentials.fly"),
    (ALEX, "essentials.home", False, "Group vip denies essentials.*"),
    (ALEX, "essentials.back", False, "Group vip denies essentials.*"),
    (SAM, "essentials.home", True, "Group staff grants essentials.home"),
    (SAM, "essentials.fly", True, "Group vip grants essentials.fly"),
    (NOBODY, "essentials.fly", False, "Group default denies essentials.fly"),
    (NOBODY, "essentials.home", True, "Group default grants essentials.home"),
])
assert groups_of(WEIGHT, SAM) == [("staff", 20, 1, 0), ("vip", 10, 1, 1), ("default", 0, 1, 2)]
assert groups_of(WEIGHT, NOBODY) == [("default", 0, 1, 0)]

OVER_DEPTH = '''
[groups.default]

[groups.builder]
inherits = ["helper"]
nodes = ["worldedit.wand"]

[groups.helper]
weight = 50
nodes = ["-worldedit.*"]

[players.%s]
groups = ["builder"]
''' % ALEX
judge(OVER_DEPTH, [
    (ALEX, "worldedit.wand", False, "Group helper denies worldedit.*"),
])
assert groups_of(OVER_DEPTH, ALEX) == [("helper", 50, 2, 0), ("builder", 0, 1, 1), ("default", 0, 1, 1)]
print("search orders: as expected")

## The depth of inheritance

At the same weight a closer group comes first. A group the player is in directly has depth 1, the
default group included, and every step of inheritance adds one along the shortest path, so a group
that the player also holds directly moves up to depth 1.

In [ ]:
DEPTH = '''
[groups.default]
nodes = ["essentials.home"]

[groups.member]
inherits = ["base"]
nodes = ["essentials.fly"]

[groups.base]
inherits = ["root"]
nodes = ["-essentials.fly", "essentials.warp"]

[groups.root]
nodes = ["-essentials.warp", "essentials.back"]

[players.%s]
groups = ["member"]

[players.%s]
groups = ["member", "root"]
''' % (ALEX, SAM)
judge(DEPTH, [
    (ALEX, "essentials.fly", True, "Group member grants essentials.fly"),
    (ALEX, "essentials.warp", True, "Group base grants essentials.warp"),
    (ALEX, "essentials.back", True, "Group root grants essentials.back"),
    (ALEX, "essentials.home", True, "Group default grants essentials.home"),
    (SAM, "essentials.warp", False, "Group root denies essentials.warp"),
    (SAM, "essentials.fly", True, "Group member grants essentials.fly"),
])
assert groups_of(DEPTH, ALEX) == [("default", 0, 1, 0), ("member", 0, 1, 0), ("base", 0, 2, 1), ("root", 0, 3, 2)]
assert groups_of(DEPTH, SAM) == [("default", 0, 1, 0), ("member", 0, 1, 0), ("root", 0, 1, 0), ("base", 0, 2, 1)]
print("search orders: as expected")

## Groups of one rank that disagree

Groups of the same weight and depth do not outrank each other. Each decides on its own, and when
they disagree, the deny wins, also against a grant that names the node more specifically. When
they agree, the first of the rank by name gives the reason.

In [ ]:
RANK = '''
[groups.default]

[groups.blue]
nodes = ["-essentials.fly", "-essentials.warp.*"]

[groups.red]
nodes = ["essentials.fly", "essentials.warp.secret", "essentials.home"]

[groups.green]
nodes = ["essentials.home"]

[players.%s]
groups = ["red", "green", "blue"]
''' % ALEX
judge(RANK, [
    (ALEX, "essentials.fly", False, "Group blue denies essentials.fly and group red of the same rank grants essentials.fly, and the deny wins"),
    (ALEX, "essentials.warp.secret", False, "Group blue denies essentials.warp.* and group red of the same rank grants essentials.warp.secret, and the deny wins"),
    (ALEX, "essentials.warp.list", False, "Group blue denies essentials.warp.*"),
    (ALEX, "essentials.home", True, "Group green grants essentials.home"),
])
assert groups_of(RANK, ALEX) == [("blue", 0, 1, 0), ("default", 0, 1, 0), ("green", 0, 1, 0), ("red", 0, 1, 0)]
print("search order: as expected")

## The default group

Every player is in the default group: a player the configuration does not name, a player with
groups of their own, and a player who names it. What the default group inherits comes at depth 2.

In [ ]:
DEFAULT = '''
[groups.default]
inherits = ["base"]
nodes = ["essentials.home"]

[groups.base]
nodes = ["essentials.list"]

[groups.vip]
weight = 10
nodes = ["essentials.fly"]

[players.%s]
groups = ["vip"]

[players.%s]
groups = ["default"]
nodes = []
''' % (ALEX, SAM)
judge(DEFAULT, [
    (NOBODY, "essentials.home", True, "Group default grants essentials.home"),
    (NOBODY, "essentials.list", True, "Group base grants essentials.list"),
    (NOBODY, "essentials.fly", None, "No rule names essentials.fly"),
    (ALEX, "essentials.home", True, "Group default grants essentials.home"),
    (ALEX, "essentials.fly", True, "Group vip grants essentials.fly"),
    (SAM, "essentials.home", True, "Group default grants essentials.home"),
    (SAM, "essentials.list", True, "Group base grants essentials.list"),
])
assert groups_of(DEFAULT, NOBODY) == [("default", 0, 1, 0), ("base", 0, 2, 1)]
assert groups_of(DEFAULT, ALEX) == [("vip", 10, 1, 0), ("default", 0, 1, 1), ("base", 0, 2, 2)]
assert groups_of(DEFAULT, SAM) == [("default", 0, 1, 0), ("base", 0, 2, 1)]
print("search orders: as expected")

## Unknown nodes and bad input at a check

A node that no rule names is undefined. A check that names no UUID, or a node that is not one
concrete node, is undefined as well, with a reason that says what is wrong. A check reads a UUID in
either case, so the UUID of Alex in upper case finds the nodes of Alex.

In [ ]:
CHECKS = '''
[groups.default]
nodes = ["essentials.home"]

[players.%s]
nodes = ["essentials.fly"]
''' % ALEX
judge(CHECKS, [
    (NOBODY, "golem.unknown", None, "No rule names golem.unknown"),
    ("steve", "essentials.home", None, 'The player "steve" is not a UUID'),
    ("a1e5a1e5000040008000000000000001", "essentials.home", None, 'The player "a1e5a1e5000040008000000000000001" is not a UUID'),
    ("a1e5a1e5-0000-4000-8000-00000000000g", "essentials.home", None, 'The player "a1e5a1e5-0000-4000-8000-00000000000g" is not a UUID'),
    (ALEX.upper(), "essentials.fly", True, "The player grants essentials.fly"),
    (NOBODY, "Essentials.home", None, "The node \"Essentials.home\" holds the character 'E', but a segment takes only a to z, 0 to 9, _ and -"),
    (NOBODY, "essentials.*", None, 'The node "essentials.*" is a wildcard, and a check names one node'),
    (NOBODY, "*", None, 'The node "*" is a wildcard, and a check names one node'),
    (NOBODY, "-essentials.home", None, 'The node "-essentials.home" starts with -, which marks a deny in the configuration'),
    (NOBODY, "", None, 'The node "" is empty'),
    (NOBODY, "essentials..home", None, 'The node "essentials..home" holds an empty segment'),
])

## Every refusal of the configuration

Loading refuses a configuration that breaks a rule of the format, with exit status 1, nothing on
standard output and one line on standard error that names where. Each case is a configuration and
the exact message after the path of the file.

In [ ]:
U = ALEX
REFUSALS = [
    ("TOML syntax", "[groups.default\n",
     "line 1, column 16: invalid table header, expected `.`, `]`"),
    ("unknown key at the top", "[group.default]\n",
     "line 1, column 2: unknown field `group`, expected `groups` or `players`"),
    ("unknown key of a group", "[groups.default]\nweigth = 1\n",
     "line 2, column 1: unknown field `weigth`, expected one of `weight`, `inherits`, `nodes`"),
    ("unknown key of a player", "[groups.default]\n[players.%s]\ngroup = []\n" % U,
     "line 3, column 1: unknown field `group`, expected `groups` or `nodes`"),
    ("weight not an integer", "[groups.default]\nweight = \"high\"\n",
     "line 2, column 10: invalid type: string \"high\", expected i32"),
    ("weight out of range", "[groups.default]\nweight = 3000000000\n",
     "line 2, column 10: invalid value: integer `3000000000`, expected i32"),
    ("nodes not a list", "[groups.default]\nnodes = \"essentials.home\"\n",
     "line 2, column 9: invalid type: string \"essentials.home\", expected a sequence"),
    ("an empty file", "",
     "groups.default: the group is missing, and every player is in it"),
    ("no default group", "[groups.admin]\nnodes = [\"*\"]\n",
     "groups.default: the group is missing, and every player is in it"),
    ("group name in upper case", "[groups.default]\n[groups.Admin]\n",
     "groups.Admin: the group name \"Admin\" holds the character 'A', but a segment takes only a to z, 0 to 9, _ and -"),
    ("group name with a dot", "[groups.default]\n[groups.\"a.b\"]\n",
     "groups.\"a.b\": the group name \"a.b\" holds a dot, and a group name is one segment"),
]
for name, node, message in [
    ("node in upper case", "Essentials.home", "holds the character 'E', but a segment takes only a to z, 0 to 9, _ and -"),
    ("node with an empty segment", "essentials..home", "holds an empty segment"),
    ("node with a trailing dot", "essentials.", "holds an empty segment"),
    ("node with a leading dot", ".essentials", "holds an empty segment"),
    ("wildcard inside a node", "essentials.*.home", "holds a * that is not a wildcard, which is * alone or .* at the end"),
    ("wildcard glued to a segment", "essentials*", "holds a * that is not a wildcard, which is * alone or .* at the end"),
    ("wildcard at the front", "*.home", "holds a * that is not a wildcard, which is * alone or .* at the end"),
    ("wildcard with no prefix", ".*", "holds an empty segment"),
    ("deny of nothing", "-", "is empty"),
    ("empty node", "", "is empty"),
    ("deny twice", "--essentials.home", "holds the segment -essentials, which starts with -, but a segment starts with a to z, 0 to 9 or _"),
    ("node with a bang", "essentials.home!", "holds the character '!', but a segment takes only a to z, 0 to 9, _ and -"),
    ("node with a space", "essentials. home", "holds the character ' ', but a segment takes only a to z, 0 to 9, _ and -"),
]:
    REFUSALS.append(("group " + name, "[groups.default]\nnodes = [%s]\n" % json.dumps(node),
                     "groups.default.nodes: the node %s %s" % (json.dumps(node), message)))
REFUSALS += [
    ("node named twice", "[groups.default]\nnodes = [\"essentials.home\", \"essentials.home\"]\n",
     "groups.default.nodes: the node \"essentials.home\" is named twice"),
    ("inherits an unknown group", "[groups.default]\n[groups.admin]\ninherits = [\"moderator\"]\n",
     "groups.admin.inherits: the group \"moderator\" is not defined"),
    ("inherits a group twice", "[groups.default]\n[groups.admin]\ninherits = [\"default\", \"default\"]\n",
     "groups.admin.inherits: the group \"default\" is named twice"),
    ("inherits itself", "[groups.default]\n[groups.a]\ninherits = [\"a\"]\n",
     "groups.a.inherits: the group inherits from itself"),
    ("a cycle of two", "[groups.default]\n[groups.a]\ninherits = [\"b\"]\n[groups.b]\ninherits = [\"a\"]\n",
     "groups.b.inherits: the inheritance a, b, a is a cycle"),
    ("a cycle of three", "[groups.default]\n[groups.a]\ninherits = [\"b\"]\n[groups.b]\ninherits = [\"c\"]\n[groups.c]\ninherits = [\"a\"]\n",
     "groups.c.inherits: the inheritance a, b, c, a is a cycle"),
    ("a cycle behind a group", "[groups.default]\ninherits = [\"a\"]\n[groups.a]\ninherits = [\"b\"]\n[groups.b]\ninherits = [\"c\"]\n[groups.c]\ninherits = [\"b\"]\n",
     "groups.c.inherits: the inheritance b, c, b is a cycle"),
    ("a player that is a name", "[groups.default]\n[players.steve]\n",
     "players.steve: the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001"),
    ("a player in upper case", "[groups.default]\n[players.%s]\n" % U.upper(),
     "players.%s: the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001" % U.upper()),
    ("a player without dashes", "[groups.default]\n[players.%s]\n" % U.replace("-", ""),
     "players.%s: the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001" % U.replace("-", "")),
    ("a player with the dashes moved", "[groups.default]\n[players.a1e5a1e50-000-4000-8000-000000000001]\n",
     "players.a1e5a1e50-000-4000-8000-000000000001: the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001"),
    ("a player in an unknown group", "[groups.default]\n[players.%s]\ngroups = [\"vip\"]\n" % U,
     "players.%s.groups: the group \"vip\" is not defined" % U),
    ("a player in a group twice", "[groups.default]\n[groups.vip]\n[players.%s]\ngroups = [\"vip\", \"vip\"]\n" % U,
     "players.%s.groups: the group \"vip\" is named twice" % U),
    ("a player node of bad syntax", "[groups.default]\n[players.%s]\nnodes = [\"Essentials.fly\"]\n" % U,
     "players.%s.nodes: the node \"Essentials.fly\" holds the character 'E', but a segment takes only a to z, 0 to 9, _ and -" % U),
    ("a player node named twice", "[groups.default]\n[players.%s]\nnodes = [\"-essentials.fly\", \"-essentials.fly\"]\n" % U,
     "players.%s.nodes: the node \"-essentials.fly\" is named twice" % U),
]

wrong = []
for name, text, expected in REFUSALS:
    said = refusal(text)
    COUNTS["refusals"] += 1
    ok = said == expected
    print("%-3s %-32s %s" % ("ok" if ok else "BAD", name, said))
    if not ok:
        wrong.append("%s:\n  expected %s\n  got      %s" % (name, expected, said))
assert not wrong, "\n".join(wrong)
print("refusals:", len(REFUSALS))

## A long chain of inheritance

Loading searches the inheritance with a stack of its own, so the default group may inherit through
a chain of `long_chain` groups, a check reaches the end of it, and the depth of the last group is
the length of the chain plus one. The same chain closed into a cycle is refused with every group of
the cycle named.

In [ ]:
lines = ["[groups.default]", "inherits = [\"g000000\"]"]
for index in range(long_chain):
    lines.append("[groups.g%06d]" % index)
    lines.append("inherits = [\"g%06d\"]" % (index + 1) if index + 1 < long_chain else "nodes = [\"essentials.home\"]")
CHAIN = "\n".join(lines) + "\n"
last = "g%06d" % (long_chain - 1)
judge(CHAIN, [(NOBODY, "essentials.home", True, "Group %s grants essentials.home" % last)])
order = groups_of(CHAIN, NOBODY)
assert len(order) == long_chain + 1, len(order)
assert order[0] == ("default", 0, 1, 0) and order[-1] == (last, 0, long_chain + 1, long_chain), (order[0], order[-1])

said = refusal(CHAIN.replace("nodes = [\"essentials.home\"]", "inherits = [\"g000000\"]"))
COUNTS["refusals"] += 1
cycle = ", ".join(["g%06d" % index for index in range(long_chain)] + ["g000000"])
assert said == "groups.%s.inherits: the inheritance %s is a cycle" % (last, cycle), said[:300]
print("groups in the search order:", len(order), "the last at depth", order[-1][2])
print("the cycle:", said[:72], "...", said[-40:])

## A configuration with every kind of key

The counterpart of the refusals: a configuration that uses every key, a deny, both kinds of
wildcard, inheritance over two steps, a player with groups and nodes and a player with neither
loads, and the groups of each player come out in the search order.

In [ ]:
FULL = '''
[groups.default]
weight = 0
inherits = []
nodes = ["essentials.home", "-essentials.home.secret", "essentials.signs.use.*"]

[groups.vip]
weight = 10
inherits = ["member"]
nodes = ["essentials.fly"]

[groups.member]
weight = 5
nodes = ["*", "-golem.*"]

[players.%s]
groups = ["vip"]
nodes = ["-essentials.fly"]

[players.%s]
''' % (ALEX, SAM)
judge(FULL, [
    (ALEX, "essentials.fly", False, "The player denies essentials.fly"),
    (ALEX, "golem.perm", False, "Group member denies golem.*"),
    (ALEX, "essentials.home.secret", True, "Group member grants *"),
    (SAM, "essentials.home.secret", False, "Group default denies essentials.home.secret"),
    (SAM, "essentials.signs.use.buy", True, "Group default grants essentials.signs.use.*"),
])
assert groups_of(FULL, ALEX) == [("vip", 10, 1, 0), ("member", 5, 2, 1), ("default", 0, 1, 2)]
assert groups_of(FULL, SAM) == [("default", 0, 1, 0)]
print("search orders: as expected")

## The seed of SkyTekx

`config/permissions.toml` must load, its group default must hold exactly the 39 nodes that the
wiki of the server lists for the group default of LuckPerms on the Mac mini, and a player that the
store does not name must hold each of them, the signs through their wildcard. The admin commands
stay out of reach of such a player and are granted to a player of the group admin, which the cell
adds to a copy of the seed with a made up UUID.

In [ ]:
# The group default of the live server, from okf/modpack/plugins.md of skytekx3-1201-server,
# section Permissions, each with the prefix essentials.
WIKI_DEFAULT = [
    "tpa", "tpaccept", "tpdeny", "tpahere", "tpacancel", "tpauto", "home", "home.bed",
    "sethome", "sethome.multiple", "delhome", "back", "back.ondeath", "spawn", "warp",
    "warp.list", "msg", "mail", "mail.send", "ignore", "me", "afk", "list", "seen", "realname",
    "motd", "rules", "help", "helpop", "ping", "getpos", "compass", "depth", "suicide",
    "balance", "balancetop", "pay", "worth", "signs.use.*",
]
EXPECTED = ["essentials." + node for node in WIKI_DEFAULT]
assert len(EXPECTED) == 39 and len(set(EXPECTED)) == 39, "the wiki list holds %d nodes" % len(EXPECTED)

result = devtool("permissions-groups", "--config", SEED, "--player", NOBODY)
assert result.returncode == 0, "the seed does not load: " + result.stderr
seed_groups = json.loads(result.stdout)["groups"]
assert [g["group"] for g in seed_groups] == ["default"], seed_groups
seed_nodes = seed_groups[0]["nodes"]
print("seed default:", len(seed_nodes), "nodes")
assert len(seed_nodes) == len(set(seed_nodes)) == 39, "the seed default holds %d nodes" % len(seed_nodes)
missing, extra = sorted(set(EXPECTED) - set(seed_nodes)), sorted(set(seed_nodes) - set(EXPECTED))
assert not missing and not extra, "missing %s, extra %s" % (missing, extra)

concrete = [node for node in EXPECTED if not node.endswith(".*")]
SIGNS = ["essentials.signs.use.buy", "essentials.signs.use.sell", "essentials.signs.use.warp", "essentials.signs.use.balance"]
judge(SEED, [(NOBODY, node, True, "Group default grants " + node) for node in concrete]
      + [(NOBODY, node, True, "Group default grants essentials.signs.use.*") for node in SIGNS])
print("granted to a default player:", len(concrete), "nodes and", len(SIGNS), "signs")

In [ ]:
ADMIN_ESSENTIALS = [
    "essentials.setspawn", "essentials.setwarp", "essentials.delwarp", "essentials.eco",
    "essentials.gamemode", "essentials.kick", "essentials.ban", "essentials.tpr",
    "essentials.home.others", "essentials.signs.create.buy",
]
ADMIN_GOLEM = ["golem.perm"]
judge(SEED, [(NOBODY, node, None, "No rule names " + node) for node in ADMIN_ESSENTIALS + ADMIN_GOLEM])

seed_with_admin = write(SEED.read_text(encoding="utf-8") + "\n[players.%s]\ngroups = [\"admin\"]\n" % ALEX)
judge(seed_with_admin,
      [(ALEX, node, True, "Group admin grants essentials.*") for node in ADMIN_ESSENTIALS]
      + [(ALEX, node, True, "Group admin grants golem.*") for node in ADMIN_GOLEM]
      + [(ALEX, node, True, "Group admin grants essentials.*") for node in concrete[:3]]
      + [(NOBODY, "essentials.setspawn", None, "No rule names essentials.setspawn")])
assert groups_of(seed_with_admin, ALEX) == [("admin", 100, 1, 0), ("default", 0, 1, 1)]
print("admin commands refused to a default player and granted to an admin:", len(ADMIN_ESSENTIALS + ADMIN_GOLEM))

## The cost of a check

A check runs on the server thread, so it must not grow with the configuration. The cell times the
same check of a player in three groups once in a configuration with that player alone and once in
one with `large_groups` groups and `large_players` players, through the form of the check that
builds no reason, and holds both to the bound of the parameters cell.

In [ ]:
import time


def configuration(groups, players):
    lines = ["[groups.default]", "nodes = [\"essentials.home\", \"essentials.signs.use.*\"]"]
    for index in range(groups):
        nodes = ["plugin%03d.node%02d" % (index, n) for n in range(20)] + ["-plugin%03d.secret.*" % index]
        lines += ["", "[groups.g%03d]" % index, "weight = %d" % (index % 7), "nodes = %s" % json.dumps(nodes)]
    for index in range(players):
        uuid = "%08x-0000-4000-8000-%012x" % (index, index)
        member = [index % groups, (index * 7 + 1) % groups, (index * 13 + 2) % groups]
        lines += ["", "[players.%s]" % uuid, "groups = %s" % json.dumps(sorted({"g%03d" % g for g in member}))]
        lines += ["nodes = %s" % json.dumps(["-plugin%03d.node%02d" % (member[0], n) for n in range(3)])]
    return "\n".join(lines) + "\n"


PLAYER0 = "%08x-0000-4000-8000-%012x" % (0, 0)
NODES = ["essentials.home", "essentials.signs.use.buy", "plugin001.node05", "plugin002.secret.x", "plugin000.node01", "essentials.setspawn"]
small = write(configuration(3, 1))
large = write(configuration(large_groups, large_players))
timing = {}
for name, path in (("small", small), ("large", large)):
    started = time.monotonic()
    out = decide(path, PLAYER0, NODES, extra=["--repeat", repeat])
    timing[name] = out["nanos_per_check"]
    print("%-5s %7d bytes, %6.0f ns per check over %d checks, %.2f s for the run" % (
        name, path.stat().st_size, out["nanos_per_check"], repeat * len(NODES), time.monotonic() - started))
    assert [d["value"] for d in out["decisions"]] == [True, True, True, False, False, None], out["decisions"]
growth = timing["large"] / timing["small"]
print("growth: %.2f" % growth)
assert max(timing.values()) < max_nanos_per_check, "a check takes %.0f ns" % max(timing.values())
assert growth < max_growth, "a check in the large configuration is %.1f times slower" % growth

## Verdict

A run that reaches this cell passed every case.

In [ ]:
import shutil

shutil.rmtree(SCRATCH)
print("verdict: %d decisions, %d refusals and %d search orders as expected, the seed grants the %d nodes of the wiki" % (
    COUNTS["decisions"], COUNTS["refusals"], COUNTS["group lists"], len(EXPECTED)))